# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [ ]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item
import random
from collections import defaultdict


load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [26]:
LITE_MODE = True

In [27]:
username = "frex1"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 20,000 items
title='LFOEwpp7 Coffee Pod Filter, 6Pcs Reusable Coffee Pod Filters Mesh Holder Replacements for Keurig K-Cup' category='Appliances' price=3.07 full='LFOEwpp7 Coffee Pod Filter, 6Pcs Reusable Coffee Pod Filters Mesh Holder Replacements for Keurig K-Cup\n[\'Package Includes:\', \'6 x Coffee Pod Filters1 x Plastic Spoon\', \'Specifications:\', \'Type: Coffee Pod Filter Color: Black Material: ABS, 304 Stainless Steel Features: Easy to Use, Durable, Good Helper Size(Upper Dia. x Bottom Dia. x H): 5cm x 3.6cm x 4.6cm/1.97" x 1.42" x 1.81" (Approx.)\', \'Notes:\', "Due to the light and screen setting difference, the item\'s color may be slightly different from the pictures. Please allow slight dimension difference due to different manual measurement.", \'LFOEwpp7 is a new seller on Amazon, every order will encourage us to do better. Feedback and Product Review will help us to get more impression. So We will appreciate every customer to share their purchasing experience wi

In [28]:
items[2].id

In [29]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [30]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [31]:
print(items[0].full)

LFOEwpp7 Coffee Pod Filter, 6Pcs Reusable Coffee Pod Filters Mesh Holder Replacements for Keurig K-Cup
['Package Includes:', '6 x Coffee Pod Filters1 x Plastic Spoon', 'Specifications:', 'Type: Coffee Pod Filter Color: Black Material: ABS, 304 Stainless Steel Features: Easy to Use, Durable, Good Helper Size(Upper Dia. x Bottom Dia. x H): 5cm x 3.6cm x 4.6cm/1.97" x 1.42" x 1.81" (Approx.)', 'Notes:', "Due to the light and screen setting difference, the item's color may be slightly different from the pictures. Please allow slight dimension difference due to different manual measurement.", 'LFOEwpp7 is a new seller on Amazon, every order will encourage us to do better. Feedback and Product Review will help us to get more impression. So We will appreciate every customer to share their purchasing experience with others very much.']
['✨ Perfect gift for coffee lover. Reusable, environmental protection, saving, good filtering effect. Lower your costs, double your value and increase your pers

In [32]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Reusable Coffee Pod Filters – 6 Pack  
Category: Kitchen & Dining  
Brand: LFOEwpp7  
Description: Durable, stainless steel mesh coffee pod filters that fit most Keurig K-Cup models.  
Details: Easy-to-install, reusable, and heat‑resistant filters that save money and reduce waste.

Input tokens: 648
Output tokens: 83
Cost: 0.007 cents


In [33]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


### System:
Category: Home Goods
Brand: LFOEwpp7
Description: Reusable coffee pod filters for Keurig machines.
Details: Easy to use, durable, and good filtering effect.

Input tokens: 620
Output tokens: 44
Cost: 0.000 cents


In [34]:
MODEL = "openai/gpt-oss-20b"


In [35]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [36]:
items[0]

<LFOEwpp7 Coffee Pod Filter, 6Pcs Reusable Coffee Pod Filters Mesh Holder Replacements for Keurig K-Cup = $3.07>

In [37]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "LFOEwpp7 Coffee Pod Filter, 6Pcs Reusable Coffee Pod Filters Mesh Holder Replacements for Keurig K-Cup\\n[\'Package Includes:\', \'6 x Coffee Pod Filters1 x Plastic Spoon\', \'Specifications:\', \'Type: Coffee Pod Filter Color: Black Material: ABS, 304 Stainless Steel Features: Easy to Use, Durable, Good Helper Size(Upper Dia. x Bottom Dia. x H): 5cm x 3.6cm x 4.6cm/1.97\\" x 1.42\\" x 1.81\\" (Approx.)\', \'Notes:\', \\"Due to the light and screen setting difference, the item\'s color may be slightly different from the pictures. Please allow slight 

In [38]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [39]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [40]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [41]:

with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

FileCreateResponse(id='file_01kjxzv6gdfqxrjd4zbz90wjtj', bytes=2162522, created_at=1772680485, filename='0_1000.jsonl', object='file', purpose='batch', size=0, md5='aq5glvNI7CqzeRUcHFfV0w==', content_type='application/jsonl')

In [42]:
file_id = response.id
file_id

'file_01kjxzv6gdfqxrjd4zbz90wjtj'

In [43]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

BatchCreateResponse(id='batch_01kjxzvg94fryaf3jdj345k65j', completion_window='24h', created_at=1772680495, endpoint='/v1/chat/completions', input_file_id='file_01kjxzv6gdfqxrjd4zbz90wjtj', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1772766895, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=RequestCounts(completed=0, failed=0, total=0), project_id='project_01kjxvsb19ezs91mrhfa243tfx')

In [44]:
result = groq.batches.retrieve(response.id)
result

BatchRetrieveResponse(id='batch_01kjxzvg94fryaf3jdj345k65j', completion_window='24h', created_at=1772680495, endpoint='/v1/chat/completions', input_file_id='file_01kjxzv6gdfqxrjd4zbz90wjtj', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1772680505, error_file_id=None, errors=None, expired_at=None, expires_at=1772766895, failed_at=None, finalizing_at=1772680504, in_progress_at=1772680496, metadata=None, output_file_id='file_01kjxzvsd6e4r99cte41ewee5w', request_counts=RequestCounts(completed=1000, failed=0, total=1000), project_id='project_01kjxvsb19ezs91mrhfa243tfx')

In [45]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [46]:
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [47]:
print(items[0].full)

LFOEwpp7 Coffee Pod Filter, 6Pcs Reusable Coffee Pod Filters Mesh Holder Replacements for Keurig K-Cup
['Package Includes:', '6 x Coffee Pod Filters1 x Plastic Spoon', 'Specifications:', 'Type: Coffee Pod Filter Color: Black Material: ABS, 304 Stainless Steel Features: Easy to Use, Durable, Good Helper Size(Upper Dia. x Bottom Dia. x H): 5cm x 3.6cm x 4.6cm/1.97" x 1.42" x 1.81" (Approx.)', 'Notes:', "Due to the light and screen setting difference, the item's color may be slightly different from the pictures. Please allow slight dimension difference due to different manual measurement.", 'LFOEwpp7 is a new seller on Amazon, every order will encourage us to do better. Feedback and Product Review will help us to get more impression. So We will appreciate every customer to share their purchasing experience with others very much.']
['✨ Perfect gift for coffee lover. Reusable, environmental protection, saving, good filtering effect. Lower your costs, double your value and increase your pers

In [49]:
print(items[0].summary)

Title: Reusable Coffee Pod Filters Mesh Holder  
Category: Kitchen & Dining  
Brand: LFOEwpp7  
Description: Six reusable black coffee pod filters that fit most Keurig and compatible K‑Cup machines, helping reduce waste and cost.  
Details: Made from durable ABS and stainless steel mesh with a silicone O‑ring seal for a secure, heat‑resistant fit and easy cleaning.


## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [50]:
Batch.create(items, LITE_MODE)

Created 20 batches


In [51]:
Batch.run()

  0%|          | 0/20 [00:00<?, ?it/s]

Submitted 20 batches


In [55]:
Batch.fetch()

  0%|          | 0/20 [00:00<?, ?it/s]

Finished 20 of 20 batches


In [56]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [57]:
print(items[10234].summary)

Title: DIY Soprano Ukulele Kit (22‑inch)  
Category: Musical Instruments  
Brand: Generic  
Description: A beginner-friendly kit to build and decorate your own 22‑inch soprano ukulele.  
Details: Features four nylon strings, pre‑sanded painted finish, and all parts included for easy assembly.


In [58]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
username = "frex1"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

TRAIN_RATIO   = 0.90
VAL_RATIO     = 0.05
# TEST_RATIO  = 0.05 (remainder)

LITE_PER_CAT  = 2_500   # 8 cats × 2,500 = 20,000 total lite items

def stratified_split(items_list, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO):
    """Split a list of items into train/val/test preserving uniform
    category distribution (90 / 5 / 5)."""
    buckets = defaultdict(list)
    for item in items_list:
        buckets[item.category].append(item)

    train, val, test = [], [], []
    for cat, bucket in buckets.items():
        n      = len(bucket)
        n_tr   = round(n * train_ratio)
        n_val  = round(n * val_ratio)
        # remainder goes to test to avoid off-by-one drift
        train.extend(bucket[:n_tr])
        val.extend(  bucket[n_tr : n_tr + n_val])
        test.extend( bucket[n_tr + n_val :])

    random.seed(42)
    for split in [train, val, test]:
        random.shuffle(split)
    return train, val, test

def stratified_lite_split(items_list, per_cat=LITE_PER_CAT,
                          train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO):
    """Build a lite split: exactly `per_cat` items per category,
    then apply the same 90/5/5 stratified split."""
    buckets = defaultdict(list)
    for item in items_list:
        buckets[item.category].append(item)

    train, val, test = [], [], []
    for cat, bucket in buckets.items():
        lite_bucket = bucket[:per_cat]
        n_tr  = round(per_cat * train_ratio)   # 2,250
        n_val = (per_cat - n_tr) // 2          # 125
        train.extend(lite_bucket[:n_tr])
        val.extend(  lite_bucket[n_tr : n_tr + n_val])
        test.extend( lite_bucket[n_tr + n_val :])

    random.seed(42)
    for split in [train, val, test]:
        random.shuffle(split)
    return train, val, test

# Push to hub 
if LITE_MODE:
    train, val, test = stratified_split(items)
    print("=== Lite dataset (stratified 90/5/5) ===")
    print(f"  Train : {len(train):,}  |  Val : {len(val):,}  |  Test : {len(test):,}")
    Item.push_to_hub(lite, train, val, test)

else:
    # Full dataset — stratified split
    train, val, test = stratified_split(items)
    print("=== Full dataset (stratified 90/5/5) ===")
    print(f"  Train : {len(train):,}  |  Val : {len(val):,}  |  Test : {len(test):,}")
    Item.push_to_hub(full, train, val, test)

    # Lite subset — 2,500 per category, same split ratios
    train_lite, val_lite, test_lite = stratified_lite_split(items)
    print("\n=== Lite dataset (2,500/cat, stratified 90/5/5) ===")
    print(f"  Train : {len(train_lite):,}  |  Val : {len(val_lite):,}  |  Test : {len(test_lite):,}")
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)


=== Lite dataset (stratified 90/5/5) ===
  Train : 18,000  |  Val : 1,000  |  Test : 1,000


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/18 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## And here they are!

https://huggingface.co/datasets/frex1/items_lite

https://huggingface.co/datasets/frex1/items_full
